# Text + metadata pipeline (Phase 3, issue I-3)

The paper claims a combined TF-IDF + one-hot-metadata feature space (speaker, subject,
party, context, job, state), but `proposed_improvements.ipynb` never builds it -- a
paper/code mismatch. This notebook implements Option A from `claude-workspace/ISSUE_PLAN.md`
Phase 3: it actually builds the metadata feature space (`metadata_features.py`) and
combines it with the TF-IDF text features, so the claim becomes true.

Three feature sets are compared, all on the label-corrected data and the same metric set
(macro-F1 primary) from Phase 1:
1. **Text only** -- carried over from `proposed_improvements_v2.ipynb` (Chi2 + MI).
2. **Text + metadata (no speaker)** -- subject/party/state/job/context, one-hot/multi-label.
3. **Text + metadata + speaker (hashed)** -- adds speaker via a 64-dim `FeatureHasher`
   rather than raw one-hot, to report its effect explicitly without giving the model a
   speaker-identity lookup table (see `metadata_features.py` docstring for why).

The five `*_counts` credit-history columns are never used (label leakage).

**2026-08-24 update (train+valid merge):** `valid.csv` was previously loaded and scored
after model selection but never used for any decision -- a wasted split. It is now merged
into the training pool (`train = train_raw + valid_raw`) before TF-IDF/metadata fitting
and `GridSearchCV`, so tuning sees ~1,284 more labeled rows; `build_metadata_features` was
correspondingly simplified from a 3-way (train/valid/test) to a 2-way (train/test) split.
Test stays untouched and is the only held-out split reported. The DistilBERT reference
notebook is deliberately *not* changed to match -- it still follows the official split, so
its training-data budget differs from the classical models here; this is disclosed in the
paper.

In [1]:
import re

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report
from sklearn.pipeline import Pipeline

from metadata_features import MetadataCanonicalizer, build_pipeline_transformer

Load data (corrected labels) and preprocess text exactly as in `proposed_improvements_v2.ipynb`

In [2]:
train_raw = load_and_label("train.csv")
valid_raw = load_and_label("valid.csv")
test = load_and_label("test.csv")

# Merge train+valid into one fitting pool (see the 2026-08-24 note above); test
# stays untouched and is the only held-out split reported below.
train = pd.concat([train_raw, valid_raw], ignore_index=True)

balance = train["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
assert 0.35 < balance["fake"] < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)


train["clean_text"] = train["Statement"].apply(preprocess)
test["clean_text"] = test["Statement"].apply(preprocess)

y_train, y_test = train["Label"], test["Label"]

Feature building moved into the model Pipelines (CR-7): TF-IDF text features use the same parameters as the text-only pipeline, but are refit inside every CV fold.

In [3]:
# Features are no longer built here, once, on the whole pool. Each model's Pipeline
# (next cells) starts with the metadata canonicalizer and the TF-IDF + metadata
# transformer, so GridSearchCV refits all of them inside every CV fold (CR-7).
y_train, y_test = train["Label"], test["Label"]

Metadata features -- no speaker, and with hashed speaker. The canonicalizer and encoders are Pipeline steps, so they are also refit inside every CV fold (CR-7).

In [4]:
def make_features(include_speaker):
    return Pipeline([
        ("canon", MetadataCanonicalizer()),
        ("features", build_pipeline_transformer(use_metadata=True, include_speaker=include_speaker)),
    ])


feature_sets = {
    "Text + metadata": False,
    "Text + metadata + speaker (hashed)": True,
}

GridSearch + evaluation -- same objective (macro-F1) and models as the text-only proposed pipeline

In [5]:
def train_and_evaluate(model, param_grid, include_speaker, X_train, y_train, X_test, y_test):
    pipe = Pipeline([("prep", make_features(include_speaker)), ("clf", model)])
    prefixed_grid = {f"clf__{k}": v for k, v in param_grid.items()}
    grid = GridSearchCV(pipe, prefixed_grid, cv=5, scoring="f1_macro", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    test_metrics = evaluate_full(y_test, y_pred)
    cv_stats = {
        "cv_mean": grid.cv_results_["mean_test_score"][grid.best_index_],
        "cv_std": grid.cv_results_["std_test_score"][grid.best_index_],
    }
    return best_model, grid.best_params_, test_metrics, cv_stats, y_pred


def make_models():
    return [
        (
            "Logistic Regression",
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "solver": ["liblinear"], "class_weight": [None, "balanced"]},
        ),
        (
            "SVM",
            LinearSVC(random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "class_weight": [None, "balanced"]},
        ),
        ("Naive Bayes", MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}),
        (
            "Random Forest",
            RandomForestClassifier(random_state=RANDOM_STATE),
            {"n_estimators": [100, 200], "max_depth": [None, 10], "min_samples_split": [2, 5]},
        ),
        (
            "XGBoost",
            XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
            {"n_estimators": [100, 200], "max_depth": [3, 6], "learning_rate": [0.01, 0.1]},
        ),
    ]

In [6]:
rows = []
predictions = pd.DataFrame({"ID": test["ID"].values, "y_true": y_test.values})
for feature_set_name, include_speaker in feature_sets.items():
    for name, model, params in make_models():
        print(f"\nTraining {name} on [{feature_set_name}] (features refit inside each CV fold)...")
        best_model, best_params, test_m, cv_stats, y_pred = train_and_evaluate(
            model, params, include_speaker, train, y_train, test, y_test
        )
        print("Best params:", best_params)
        print_report(f"{name} [{feature_set_name}]", y_test, y_pred)
        predictions[f"pred__{name}__{feature_set_name}"] = y_pred
        rows.append(
            {
                "Pipeline": "Proposed",
                "Method": feature_set_name,
                "Model": name,
                "Test Accuracy": test_m["accuracy"],
                "Test Macro-F1": test_m["macro_f1"],
                "Test Fake Precision": test_m["fake_precision"],
                "Test Fake Recall": test_m["fake_recall"],
                "Test Fake F1": test_m["fake_f1"],
                "Test Real F1": test_m["real_f1"],
                "Test Confusion Matrix": test_m["confusion_matrix"],
                "CV Macro-F1 Mean": cv_stats["cv_mean"],
                "CV Macro-F1 Std": cv_stats["cv_std"],
                "Best Params": best_params,
            }
        )

metadata_results = pd.DataFrame(rows)
predictions.to_csv("model_comparison_test_predictions_v3_metadata.csv", index=False)


Training Logistic Regression on [Text + metadata] (features refit inside each CV fold)...


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


... (103 outputs left out)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 6, 'clf__n_estimators': 200}

XGBoost [Text + metadata + speaker (hashed)]
[[260 293]
 [160 554]]
              precision    recall  f1-score   support

        fake      0.619     0.470     0.534       553
        real      0.654     0.776     0.710       714

    accuracy                          0.642      1267
   macro avg      0.637     0.623     0.622      1267
weighted avg      0.639     0.642     0.633      1267



Merge with the text-only (Chi2/MI) and baseline/Dummy results from Phase 1 -- one comparable
table, same metric set throughout (I-4). This is also the raw material for the Phase 2
ablation (`text -> +metadata -> +feature selection -> +tuning`).

In [7]:
phase1_results = pd.read_csv("model_comparison_results_v2.csv")

all_results = pd.concat([phase1_results, metadata_results], ignore_index=True)
all_results = all_results.sort_values("Test Macro-F1", ascending=False)
all_results.to_csv("model_comparison_results_v3_metadata.csv", index=False)
all_results[["Pipeline", "Method", "Model", "Test Accuracy", "Test Macro-F1", "Test Fake F1"]]

,Pipeline,Method,Model,Test Accuracy,Test Macro-F1,Test Fake F1
20,Proposed,Text + metadata,Naive Bayes,0.664562,0.654900,0.597156
25,Proposed,Text + metadata + speaker (hashed),Naive Bayes,0.657459,0.647877,0.589792
24,Proposed,Text + metadata + speaker (hashed),SVM,0.641673,0.634376,0.582721
23,Proposed,Text + metadata + speaker (hashed),Logistic Regression,0.639305,0.631875,0.579577
21,Proposed,Text + metadata,Random Forest,0.650355,0.625566,0.529224
26,Proposed,Text + metadata + speaker (hashed),Random Forest,0.647987,0.623842,0.528541
18,Proposed,Text + metadata,Logistic Regression,0.629834,0.622884,0.571689
27,Proposed,Text + metadata + speaker (hashed),XGBoost,0.642463,0.622116,0.534430
19,Proposed,Text + metadata,SVM,0.629045,0.621998,0.570384
0,Proposed,Mutual Information,Logistic Regression,0.623520,0.620338,0.585578


Isolate the metadata effect: best text-only vs. best text+metadata vs. best text+metadata+speaker,
same model family where possible.

In [8]:
methods_of_interest = [
    "Chi-square",
    "Mutual Information",
    "Text + metadata",
    "Text + metadata + speaker (hashed)",
]
summary = (
    all_results[all_results["Method"].isin(methods_of_interest)]
    .sort_values("Test Macro-F1", ascending=False)
    .groupby("Method", sort=False)
    .first()[["Model", "Test Accuracy", "Test Macro-F1", "Test Fake F1"]]
)
summary.reindex(methods_of_interest)

,Model,Test Accuracy,Test Macro-F1,Test Fake F1
Method,,,,
Chi-square,Logistic Regression,0.617206,0.614709,0.583691
Mutual Information,Logistic Regression,0.623520,0.620338,0.585578
Text + metadata,Naive Bayes,0.664562,0.654900,0.597156
Text + metadata + speaker (hashed),Naive Bayes,0.657459,0.647877,0.589792
